# 02 · Clean the data and review the changes

This notebook applies the agreed first-pass rules using `src/clean_data.py`.
It replaces impossible numeric cells with missing values, labels missing categories `Unknown`,
and retains every user. Numeric gaps remain missing; activity outliers are flagged without capping.

Source rules: `docs/case/Datathon_2026_Case_Addendum.pdf`, section 2.
The source workbook remains unchanged. Individual records and per-cell logs are not displayed here.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/clean_data.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the project repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.clean_data import run_cleaning, CATEGORIES, NUMERIC

cleaned, changes, audit, manifest = run_cleaning(ROOT)
display(Markdown((ROOT / 'reports/cleaning_summary.md').read_text()))


## Review completeness after cleaning

Categorical `Unknown` values are explicit missingness, not recovered information. Numeric missingness
increases where invalid cells were removed. Flags distinguish originally missing values from invalid ones.


In [ ]:
display(audit.loc[(audit.missing_before > 0) | (audit.invalid_before > 0)].reset_index(drop=True))
display(changes.groupby(['field', 'reason'], dropna=False).size().rename('changed_cells').to_frame())
print('Original analytical fields:', manifest['original_fields'])
print('Output fields including provenance flags:', manifest['output_fields_including_flags'])
print('Likes screening thresholds:', manifest['outlier_thresholds']['likes_per_week'])


## Check the analytical data

These summaries select original numeric fields explicitly. Pandas means and medians exclude missing
values; the `count` column records coverage. A total built from incomplete revenue data needs a stated
missing-value assumption and sensitivity checks before it can support a recommendation.


In [ ]:
display(cleaned[NUMERIC].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']])
display(pd.DataFrame({
    'unknown_users': {c: int(cleaned[c].eq('Unknown').sum()) for c in CATEGORIES},
    'unknown_pct': {c: 100 * cleaned[c].eq('Unknown').mean() for c in CATEGORIES},
}))
assert len(cleaned) == manifest['rows_before']
assert cleaned.user_id.is_unique
assert len(changes) == int(audit.changed_cells.sum())
assert not cleaned[list(CATEGORIES)].isna().any().any()
print('Checks passed. Cleaned data and detailed change logs are in data/processed/.')


## Next analysis

Use `data/processed/all_users_cleaned.csv` for segmentation and modeling. Select original features
deliberately and retain the provenance flags for sensitivity checks. Compare revenue estimates under
conservative and documented imputation scenarios. Compare activity summaries with and without the
flagged likes tail; the current output retains that tail. No final monetization recommendation is made here.
